<a href="https://colab.research.google.com/github/itsankitt231/Altrivia/blob/main/On_page_Data_Extractor_(Assessment).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install requests beautifulsoup4 pandas gspread google-auth


> Verified installation of required packages **beautifulsoup**(for scraping html data), **pandas**(for visualisation), **gspread** (for interacting with google sheet),**google-authe** (for authentication)

>> Next importing the required libraries


In [8]:
import requests
import pandas as pd
import json

from bs4 import BeautifulSoup

**Checking requests for sample url**

In [9]:
import requests

response = requests.get("https://www.pregradcampus.in")

print(response.status_code)

200


**Checking beautiful soup with html parser**

In [10]:
from bs4 import BeautifulSoup

html = requests.get("https://www.pregradcampus.in").text

soup = BeautifulSoup(html, "html.parser")

print(soup.title.text)

Pregrad | India's 1st Career Accelerator Platform 


**Checking for meta content**

In [11]:
meta = soup.find("meta", attrs={"name": "description"})

if meta:
    print(meta.get("content"))
else:
    print("Meta Description Not Found")

Accelerate your career with Pregrad. Gain essential skills, get certified with IBM, Meta, and Intuit, connect with top experts, and fast-track your success. Apply for a scholarship today!


**Checking for h1 title**

In [12]:
h1 = soup.find("h1")

if h1:
    print(h1.get_text(strip=True))
else:
    print("H1 Not Found")

Start your careerbefore Graduation


**Checking for all scripts and printing the length if exist**

In [13]:
json_ld = soup.find_all(
    "script",
    attrs={"type": "application/ld+json"}
)

print("Total JSON-LD blocks:", len(json_ld))

Total JSON-LD blocks: 0


**Checking for json text**

In [14]:
for script in json_ld:
    print(script.text)
    print("----------------------------------")

**Combining all data in a result object**

In [15]:
title=soup.title.text
result={
    "url":"https://www.pregradcampus.in",
    "title":title,
    "meta description":meta.get("content") if meta else "",
    "h1":h1.get_text(strip=True) if h1 else "",
    "json-ld count":len(json_ld)
}

**Creating a Pandas data frame to display the results**

In [16]:
df = pd.DataFrame([result])

df

,url,title,meta description,h1,json-ld count
0,https://www.pregradcampus.in,Pregrad | India's 1st Career Accelerator Platf...,Accelerate your career with Pregrad. Gain esse...,Start your careerbefore Graduation,0


**Converting the CSV File**

In [17]:
df.to_csv("seo_report.csv", index=False)

**Downloading the csv files with google.colab module**

In [18]:
from google.colab import files

files.download("seo_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Authenticate users with gmail account**

In [19]:
from google.colab import auth

auth.authenticate_user()

**Authorising Google Sheet with credentials to set gspread**

In [20]:
import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

print("Connected Successfully!")

Connected Successfully!


**Opening the gsheet files with the urls**


In [21]:
sheet = gc.open("On-page SEO Extractor (Task 2)").sheet1

print("Sheet opened successfully!")

Sheet opened successfully!


**Getting the required urls from sheet**


In [22]:
urls = sheet.col_values(1)

print(urls)

['URL', 'https://www.revvgrowth.com', 'https://pregradcampus.in', 'https://www.python.org', 'https://developers.google.com/search', 'https://openai.com', 'https://www.wikipedia.org']


**Removing the headers**


In [23]:
urls = sheet.col_values(1)[1:]

print(urls)

['https://www.revvgrowth.com', 'https://pregradcampus.in', 'https://www.python.org', 'https://developers.google.com/search', 'https://openai.com', 'https://www.wikipedia.org']


In [24]:
results = []

**Encapsulating the data extractor with exception handling**


In [25]:
for url in urls:

    print("="*60)
    print("Processing:", url)

    try:

        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=15
        )

        status = response.status_code

        soup = BeautifulSoup(response.text, "html.parser")

        title = (
            soup.title.text.strip()
            if soup.title
            else "Not Found"
        )

        meta = soup.find(
            "meta",
            attrs={"name": "description"}
        )

        meta_description = (
            meta.get("content")
            if meta
            else "Not Found"
        )

        h1 = soup.find("h1")

        h1_text = (
            h1.get_text(strip=True)
            if h1
            else "Not Found"
        )


        canonical = soup.find(
            "link",
            rel="canonical"
        )

        canonical_url = (
            canonical.get("href")
            if canonical
            else "Not Found"
        )

        robots = soup.find(
            "meta",
            attrs={"name": "robots"}
        )

        robots_content = (
            robots.get("content")
            if robots
            else "Not Found"
        )

        json_ld = soup.find_all(
            "script",
            attrs={"type": "application/ld+json"}
        )

        json_ld_count = len(json_ld)

        results.append({

            "URL": url,

            "HTTP Status": status,

            "Title": title,

            "Meta Description": meta_description,

            "H1": h1_text,

            "Canonical": canonical_url,

            "Robots": robots_content,

            "JSON-LD Blocks": json_ld_count,

            "Status": "Success"

        })

        print("✅ Success")

    except Exception as e:

        results.append({

            "URL": url,

            "HTTP Status": "",

            "Title": "",

            "Meta Description": "",

            "H1": "",

            "Canonical": "",

            "Robots": "",

            "JSON-LD Blocks": "",

            "Status": str(e)

        })

        print("❌ Error:", e)

Processing: https://www.revvgrowth.com
✅ Success
Processing: https://pregradcampus.in
❌ Error: HTTPSConnectionPool(host='pregradcampus.in', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)')))
Processing: https://www.python.org
✅ Success
Processing: https://developers.google.com/search
✅ Success
Processing: https://openai.com
✅ Success
Processing: https://www.wikipedia.org
✅ Success


**Load dataframe**

In [26]:
import pandas as pd
df=pd.DataFrame(results)
df

,URL,HTTP Status,Title,Meta Description,H1,Canonical,Robots,JSON-LD Blocks,Status
0,https://www.revvgrowth.com,200,SaaS Marketing Agency | SaaS Marketing Services,Not Found,"SaaS Marketing Agency Turning SEO, AEO, and GE...",https://www.revvgrowth.com,Not Found,4,Success
1,https://pregradcampus.in,,,,,,,,"HTTPSConnectionPool(host='pregradcampus.in', p..."
2,https://www.python.org,200,Welcome to Python.org,The official home of the Python Programming La...,Intuitive Interpretation,Not Found,Not Found,1,Success
3,https://developers.google.com/search,200,Google Search Central (formerly Webmasters) | ...,Google Search Central provides SEO resources t...,GoogleSearch Central,https://developers.google.com/search,Not Found,0,Success
4,https://openai.com,403,Not Found,Not Found,Not Found,Not Found,Not Found,0,Success
5,https://www.wikipedia.org,200,Wikipedia,"Wikipedia is a free online encyclopedia, creat...",WikipediaThe Free Encyclopedia,Not Found,Not Found,0,Success


In [28]:
pd.set_option("display.max_colwidth", None)

df

,URL,HTTP Status,Title,Meta Description,H1,Canonical,Robots,JSON-LD Blocks,Status
0,https://www.revvgrowth.com,200,SaaS Marketing Agency | SaaS Marketing Services,Not Found,"SaaS Marketing Agency Turning SEO, AEO, and GEO IntoPipeline",https://www.revvgrowth.com,Not Found,4,Success
1,https://pregradcampus.in,,,,,,,,"HTTPSConnectionPool(host='pregradcampus.in', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)')))"
2,https://www.python.org,200,Welcome to Python.org,The official home of the Python Programming Language,Intuitive Interpretation,Not Found,Not Found,1,Success
3,https://developers.google.com/search,200,Google Search Central (formerly Webmasters) | Web SEO Resources | Google for Developers,Google Search Central provides SEO resources to help you get your website on Google Search. Learn how to make your website more discoverable today.,GoogleSearch Central,https://developers.google.com/search,Not Found,0,Success
4,https://openai.com,403,Not Found,Not Found,Not Found,Not Found,Not Found,0,Success
5,https://www.wikipedia.org,200,Wikipedia,"Wikipedia is a free online encyclopedia, created and edited by volunteers around the world and hosted by the Wikimedia Foundation.",WikipediaThe Free Encyclopedia,Not Found,Not Found,0,Success
